<a href="https://colab.research.google.com/github/Binay-panda19/Deep-Learning/blob/RAG/05-RAGs/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

 Convert Data to Documents

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {
        "source" : "https:/www.google.com"
    }
)

In [ ]:
sample_doc

Document(metadata={'source': 'https:/www.google.com'}, page_content='Hello World')

In [ ]:
type(sample_doc)

langchain_core.documents.base.Document

In [ ]:
# text data
from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader(file_path)

# document = loader.load()

# print(document)

In [ ]:
# pdf data
from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader(file_path)

# document = pdf_loader.load()

# print(document)

#Ingestion Pipeline

Data => Documents

In [ ]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [ ]:
def load_all_pdfs():
    folder_path = "/content/sample_data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if(filename.lower().endswith(".pdf")):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages", len(all_docs))
    return all_docs

In [ ]:
all_pdfs_documents = load_all_pdfs()

total pdfs: 14
total pages 163


Convert Documents to Chunks

In [ ]:
!pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size = 500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return  chunked_docs

In [ ]:
chunks = split_doc(all_pdfs_documents)

In [ ]:
len(chunks)

745

Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

        print("embedding dimensions :", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape :", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions : 384


/tmp/ipykernel_1402/2710365236.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions :", self.model.get_sentence_embedding_dimension())


Vector Store

In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStore:
    def __init__(self, persist_directory = "data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name

        self.collection = None

        self.client = None

        self._initialize_store()

    def _initialize_store(self,):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection=", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if(len(documents) != len(embeddings)):
            raise ValueError("documents and embeddings must be of same length")

        # store => ids, embeddings, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids = ids,
            embeddings = embeddings_list,
            documents = documents_content,
            metadatas = all_metadata
        )

        print("total documents added in collection=", len(documents_content))
        print("docs in collections:", self.collection.count())

In [ ]:
vector_store = VectorStore(persist_directory="data/vector_store2")

initialized the vector store with collection= pdf_documents
docs in collection: 0


# data => documents => chunks => embeddings => store in vector store

In [ ]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

embeddings shape : (745, 384)
total documents added in collection= 745
docs in collections: 745


In [ ]:
query_embedding = embedding_manager.generate_embeddings(["What is AVL?"])[0]

result = vector_store.collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print("DOCUMENTS:")
print(result["documents"])

print("\nDISTANCES:")
print(result["distances"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
DOCUMENTS:
[['AVL Trees\nAVL\nTree is a self-balancing BST.\nBalance\nFactor\nin\nAVL\ntree\n=\nheight(left\nSubtree)\n-\nheight(right\nSubtree)\nand\nalways equal to {-1, 0, 1}\nThere are 4 cases of rotation in an AVL Tree:\na.\nLL Case\n-\nRight Rotate\nb.\nRR Case\n-\nLeft Rotate\nc.\nLR Case\n-\nLeft Rotate\n-\nRight Rotate\nd.\nRL Case\n-\nRight Rotate\n-\nLeft Rotate\nAVL Tree Code (Insertion)\npublic\nclass\nAVLTrees\n{\nstatic\nclass\nNode\n{\nint\ndata\n,\nheight\n;\nNode\nleft\n,\nright\n;\nNode\n(\nint\ndata\n) {\nthis\n.\ndata\n=\ndata\n;\nheight\n=\n1\n;\n}\n}\npublic', '}\nreturn\nroot\n;\n//returned if AVL balanced\n}\npublic\nstatic\nvoid\npreorder\n(\nNode\nroot\n) {\nif\n(\nroot\n==\nnull\n) {\nreturn\n;\n}\nSystem\n.\nout\n.\nprint\n(\nroot\n.\ndata\n+\n" "\n);\npreorder\n(\nroot\n.\nleft\n);\npreorder\n(\nroot\n.\nright\n);\n}\npublic\nstatic\nvoid\nmain\n(\nString\n[]\nargs\n) {\nroot\n=\ninsert\n(\nroot\n,\n10\n);\nroot\n=\ninsert\n(\nr

#Retrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class Retriever:
    def __init__(self, embedding_manager, vector_store):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document,
                        "distance": distance,
                        "rank" : i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [ ]:
retriever = Retriever(embedding_manager, vector_store)

In [ ]:
results = retriever.retrieve("What is AVL?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 5 documents


In [ ]:
results

[{'id': 'doc_235c87f8-83e3-47ae-8fab-3d43c2f7731c',
  'metadata': {'creator': 'PyPDF',
   'total_pages': 7,
   'creationdate': '2026-02-14T13:35:26+00:00',
   'moddate': '2026-02-14T13:35:26+00:00',
   'source': '/content/sample_data/AVL Trees in Java.pdf',
   'page_label': '1',
   'content_length': 500,
   'page': 0,
   'doc_index': 608,
   'title': 'AVL Trees in Java',
   'producer': 'pdfcpu v0.9.1 dev'},
  'document': 'AVL Trees\nAVL\nTree is a self-balancing BST.\nBalance\nFactor\nin\nAVL\ntree\n=\nheight(left\nSubtree)\n-\nheight(right\nSubtree)\nand\nalways equal to {-1, 0, 1}\nThere are 4 cases of rotation in an AVL Tree:\na.\nLL Case\n-\nRight Rotate\nb.\nRR Case\n-\nLeft Rotate\nc.\nLR Case\n-\nLeft Rotate\n-\nRight Rotate\nd.\nRL Case\n-\nRight Rotate\n-\nLeft Rotate\nAVL Tree Code (Insertion)\npublic\nclass\nAVLTrees\n{\nstatic\nclass\nNode\n{\nint\ndata\n,\nheight\n;\nNode\nleft\n,\nright\n;\nNode\n(\nint\ndata\n) {\nthis\n.\ndata\n=\ndata\n;\nheight\n=\n1\n;\n}\n}\npublic'

#Integrate with LLMs

In [ ]:
API_KEY=""

In [ ]:
# !pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    groq_api_key=API_KEY,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [ ]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=5):

    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}
            """

    response = llm.invoke(prompt) # expeting a list as prompt
    return response

In [ ]:
answer = generate_output("What is AVL?", retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 5 documents


In [ ]:
print(answer)

content='**AVL** stands for **Adelson‑Velsky and Landis**, the inventors of this data structure.  \nAn **AVL tree** is a **self‑balancing binary search tree** (BST) in which every node maintains a **balance factor** defined as  \n\n\\[\n\\text{balance factor} = \\text{height(left subtree)} - \\text{height(right subtree)}\n\\]\n\nThe balance factor of every node is always kept within the set **{-1,\u202f0,\u202f1}**.  \nIf an insertion or deletion causes a node’s balance factor to go outside this range, the tree is re‑balanced using one of four rotations (LL, RR, LR, RL). This guarantees that the height of the AVL tree remains \\(O(\\log n)\\), providing logarithmic‑time search, insertion, and deletion operations.' additional_kwargs={'reasoning_content': 'We need to answer query "What is AVL?" using given context. Provide definition: AVL tree is a self-balancing binary search tree, balance factor etc. Summarize. Probably concise.'} response_metadata={'token_usage': {'completion_tokens':

In [ ]:
answer = generate_output("Give a breif summary of DBMS ?", retriever, llm,10)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 10 documents


In [ ]:
print(answer)

content='**Brief Summary of DBMS**\n\nA **Database Management System (DBMS)** is software that enables the creation, storage, retrieval, and management of data in a structured way while enforcing security and integrity constraints. It acts as an intermediary between applications and the operating system: applications submit data‑access requests, and the DBMS translates these into operations that the OS executes on the underlying database files. Key functions include:\n\n- **Data definition** (via DDL commands such as\u202fCREATE, ALTER, DROP, TRUNCATE, RENAME) to specify schemas and structures.  \n- **Data manipulation** (via DML commands) to insert, update, delete, and query data.  \n- **Security and access control** to ensure only authorized users can view or modify data.  \n- **Concurrency control and transaction management** to maintain consistency when multiple users access the database simultaneously.  \n- **Recovery and backup** mechanisms to protect against data loss.\n\nIn lar

In [ ]:
answer = generate_output("Give a breif summary of RAGs and photos and mindmaps
 ?", retriever, llm,10)

SyntaxError: unterminated string literal (detected at line 1) (4133815704.py, line 1)

In [ ]:
print(answer)

content='**Retrieval‑Augmented Generation (RAG) – Brief Summary**\n\nRetrieval‑Augmented Generation (RAG) combines a **retriever** that fetches relevant external documents with a **generator** (typically a large language model) that synthesizes those documents into a final answer. The overall workflow consists of three core steps:\n\n1. **Indexing** – Source documents are split into chunks, encoded into vector embeddings, and stored in a vector database.  \n2. **Retrieval** – Given a user query, the system retrieves the top‑k most semantically similar chunks.  \n3. **Generation** – The original query and the retrieved chunks are fed together to the LLM, which produces a grounded response.\n\n### Evolution of RAG Paradigms\n| Paradigm | Key Characteristics |\n|----------|----------------------|\n| **Naïve RAG** | Simple pipeline: retrieve‑then‑generate. Low cost, but vulnerable to noisy or contradictory retrieved information. |\n| **Advanced RAG** | Improves retrieval quality (e.g., fin